# Mundiales 2018, 2022 y 2026
## Preparación de datos y entrada al análisis supervisado

Trabajaremos exclusivamente con la fase de grupos. Los archivos contienen errores deliberados. No uses la base del profesor.

## Objetivos

- Perfilar datos.
- Estandarizar esquemas.
- Limpiar fechas, equipos, goles y marcadores.
- Eliminar duplicados con criterio.
- Comparar torneos mediante tasas.
- Construir variables conocidas antes de cada partido.
- Entrenar un primer clasificador y detectar fuga de información.

In [ ]:
# Imports: rutas, regex, unicode, datos, graficos y ML
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# Ruta relativa a la carpeta datos/
DATA = Path('../datos')
pd.set_option('display.max_columns', 50)

## Cargar los tres archivos

In [ ]:
# Cargar como strings para evitar conversiones prematuras
d18 = pd.read_csv(DATA / 'mundial_2018_sucio.csv', dtype=str)
d22 = pd.read_csv(DATA / 'mundial_2022_sucio.csv', dtype=str)
d26 = pd.read_csv(DATA / 'mundial_2026_sucio.csv', dtype=str)

# Vista rapida: dimensiones y primeras filas de cada archivo
for nombre, df in [('2018', d18), ('2022', d22), ('2026', d26)]:
    print(f'{nombre}: {df.shape}')
    display(df.head(2))


## Perfilado

Para cada archivo revisa:

- columnas;
- tipos;
- valores nulos;
- duplicados;
- valores únicos de grupos, fases y equipos;
- goles que no puedan convertirse directamente a número.

In [ ]:
# Perfilado: detectar problemas de calidad antes de limpiar
def perfil(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Dimensiones: {df.shape}')
    print(f'Columnas: {list(df.columns)}')
    print(f'Duplicados exactos: {df.duplicated().sum()}')
    print(f'Nulos por columna:\n{df.isnull().sum()}')
    print(f'Valores únicos en grupo: {df.iloc[:, 3].unique()}')
    print(f'Equipos local únicos: {df.iloc[:, 7].unique()}')
    print(f'Goles local (10 primeras): {df.iloc[:, 9].unique()[:10]}')

perfil(d18, '2018')
perfil(d22, '2022')
perfil(d26, '2026')

## Unificar nombres de columnas

In [ ]:
# Los tres CSVs tienen esquemas distintos: mapeamos cada uno al canonico
rename_maps = {
    2018: {
        'ID Partido': 'partido_id',
        'Año': 'mundial',
        'Fase': 'fase',
        'Grupo': 'grupo',
        'Jornada': 'jornada',
        'Fecha': 'fecha',
        'Equipo Local': 'equipo_local',
        'Equipo Visitante': 'equipo_visitante',
        'Goles Local': 'goles_local',
        'Goles Visitante': 'goles_visitante',
        'Marcador': 'marcador',
        'Anfitrión Local': 'local_es_anfitrion',
        'Fuente': 'fuente',
    },
    2022: {
        'match_id': 'partido_id',
        'WorldCup': 'mundial',
        'stage': 'fase',
        'group_name': 'grupo',
        'match_day': 'jornada',
        'date': 'fecha',
        'local': 'equipo_local',
        'visitor': 'equipo_visitante',
        'home_score': 'goles_local',
        'away_score': 'goles_visitante',
        'score_text': 'marcador',
        'home_host': 'local_es_anfitrion',
        'source_url': 'fuente',
    },
    2026: {
        'match': 'partido_id',
        'wc': 'mundial',
        'round': 'fase',
        'grp': 'grupo',
        'md': 'jornada',
        'played_on': 'fecha',
        'home': 'equipo_local',
        'away': 'equipo_visitante',
        'HG': 'goles_local',
        'AG': 'goles_visitante',
        'result_raw': 'marcador',
        'host_h': 'local_es_anfitrion',
        'host_a': 'visitante_es_anfitrion',
        'source': 'fuente',
    },
}

# Esquema canonico: 14 columnas comunes para los tres torneos
columnas_base = [
    'partido_id', 'mundial', 'fase', 'grupo', 'jornada', 'fecha',
    'equipo_local', 'equipo_visitante', 'goles_local',
    'goles_visitante', 'marcador', 'local_es_anfitrion',
    'visitante_es_anfitrion', 'fuente'
]

## Normalizar equipos

No conviene borrar acentos del nombre que se mostrará. Crea una clave auxiliar sin acentos, minúscula y sin signos para buscar en el catálogo.

In [ ]:
# Catalogo: mapea ~50 variantes a nombres canonicos (Alemania -> Germany)
catalogo = pd.read_csv(DATA / 'catalogo_equipos.csv')

# Normalizar texto: quitar acentos, signos, espacios para comparar
def clave_texto(valor):
    valor = unicodedata.normalize('NFKD', str(valor))
    valor = valor.encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'[^a-z0-9]', '', valor.lower().strip())

# Diccionario: clave normalizada -> nombre canonico
cat_map = {}
for _, row in catalogo.iterrows():
    cat_map[clave_texto(row['variante'])] = row['nombre_canonico']


## Fechas, grupos, booleanos y marcadores

In [ ]:
# Rango de fechas valido para cada torneo (ayuda a desambiguar formatos)
rangos = {
    2018: ('2018-06-14', '2018-06-28'),
    2022: ('2022-11-20', '2022-12-02'),
    2026: ('2026-06-11', '2026-06-27'),
}

# Convertir fecha: maneja 7 formatos + serial Excel (ej. 46187)
def convertir_fecha(valor, mundial):
    if pd.isna(valor) or str(valor).strip().lower() in ('n/d', 's/d', ''):
        return pd.NaT
    valor = str(valor).strip()
    # Detectar serial de Excel (numero entre 43000 y 48000)
    try:
        num = float(valor)
        if 43000 < num < 48000:
            return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(num))
    except ValueError:
        pass
    formats = [
        '%Y-%m-%d',
        '%d/%m/%Y', '%m/%d/%Y',
        '%d-%m-%y', '%m-%d-%y',
        '%b %d, %Y', '%B %d, %Y',
    ]
    start, end = rangos[mundial]
    # Probar cada formato y validar contra el rango del torneo
    for fmt in formats:
        try:
            dt = pd.to_datetime(valor, format=fmt, errors='raise')
            if pd.Timestamp(start) <= dt <= pd.Timestamp(end):
                return dt
        except (ValueError, TypeError):
            continue
    return pd.NaT

# Extraer el primer entero de un texto (maneja '5 goles', 'N/A', etc.)
def extraer_numero(valor):
    if pd.isna(valor):
        return np.nan
    m = re.search(r'(-?\d+)', str(valor))
    return int(m.group(1)) if m else np.nan

# Separar marcador en goles local/visitante (-, -, :, x, X)
def separar_marcador(valor):
    if pd.isna(valor):
        return np.nan, np.nan
    v = str(valor).strip().lower()
    if v in ('sin dato', 's/d', '', 'nan'):
        return np.nan, np.nan
    v = re.sub(r'[\s]', '', v)
    parts = re.split(r'[-–—:xX]+', v, maxsplit=1)
    if len(parts) != 2:
        return np.nan, np.nan
    try:
        return int(parts[0]), int(parts[1])
    except ValueError:
        return np.nan, np.nan

# Extraer letra de grupo A-L desde variantes como 'Grupo A', 'group-b', 'A'
def normalizar_grupo(valor):
    if pd.isna(valor):
        return np.nan
    v = str(valor).strip()
    m = re.search(
        r'[Gg]rupp?o?\s*([A-La-l])|[Gg]roup[\s-]*([A-La-l])|^([A-La-l])$',
        v
    )
    if m:
        letra = next(g for g in m.groups() if g is not None)
        return letra.upper()
    return np.nan

# Normalizar booleano: True si es 'Si', 'TRUE', '1', 'verdadero', etc.
def normalizar_booleano(valor):
    if pd.isna(valor):
        return False
    return str(valor).strip().lower() in ('si', 'sí', 'yes', 'true', '1', 'verdadero')


## Función de limpieza reproducible

In [ ]:
def limpiar_mundial(df, mundial):
    # Renombrar columnas segun el esquema canonico
    df = df.rename(columns=rename_maps[mundial])
    # Agregar columnas faltantes (ej. visitante_es_anfitrion en 2018/2022)
    for col in columnas_base:
        if col not in df.columns:
            df[col] = np.nan
    df = df[columnas_base]

    # Normalizar nombres de equipo via catalogo
    df['equipo_local'] = df['equipo_local'].apply(
        lambda x: cat_map.get(clave_texto(x), str(x).strip()) if pd.notna(x) else x
    )
    df['equipo_visitante'] = df['equipo_visitante'].apply(
        lambda x: cat_map.get(clave_texto(x), str(x).strip()) if pd.notna(x) else x
    )

    # Normalizar grupo, fecha y booleanos
    df['grupo'] = df['grupo'].apply(normalizar_grupo)
    df['fecha'] = df['fecha'].apply(lambda x: convertir_fecha(x, mundial))
    df['local_es_anfitrion'] = df['local_es_anfitrion'].apply(normalizar_booleano)
    if 'visitante_es_anfitrion' in df.columns:
        df['visitante_es_anfitrion'] = df['visitante_es_anfitrion'].apply(normalizar_booleano)
    else:
        df['visitante_es_anfitrion'] = False

    # Extraer numeros de goles (maneja '5 goles', '-1', 'N/A', etc.)
    df['goles_local'] = df['goles_local'].apply(extraer_numero)
    df['goles_visitante'] = df['goles_visitante'].apply(extraer_numero)

    # Reparar goles usando el marcador cuando falten o sean negativos
    mask_repair = (
        df['goles_local'].isna() | df['goles_visitante'].isna()
        | (df['goles_local'] < 0) | (df['goles_visitante'] < 0)
    )
    for idx in df[mask_repair].index:
        gl, gv = separar_marcador(df.loc[idx, 'marcador'])
        if pd.notna(gl) and gl >= 0:
            if pd.isna(df.loc[idx, 'goles_local']) or df.loc[idx, 'goles_local'] < 0:
                df.loc[idx, 'goles_local'] = gl
        if pd.notna(gv) and gv >= 0:
            if pd.isna(df.loc[idx, 'goles_visitante']) or df.loc[idx, 'goles_visitante'] < 0:
                df.loc[idx, 'goles_visitante'] = gv

    # Inferir grupos faltantes desde otros partidos del mismo equipo
    team_group = {}
    for _, row in df.iterrows():
        if pd.notna(row['grupo']):
            team_group[row['equipo_local']] = row['grupo']
            team_group[row['equipo_visitante']] = row['grupo']
    for idx in df[df['grupo'].isna()].index:
        for equipo in [df.loc[idx, 'equipo_local'], df.loc[idx, 'equipo_visitante']]:
            if equipo in team_group:
                df.loc[idx, 'grupo'] = team_group[equipo]
                break

    # Eliminar filas duplicadas por partido_id
    df = df.drop_duplicates(subset='partido_id', keep='first')

    # Derivar columnas de analisis
    df['goles_totales'] = df['goles_local'] + df['goles_visitante']
    df['diferencia_goles'] = df['goles_local'] - df['goles_visitante']
    df['resultado_local'] = np.where(
        df['goles_local'] > df['goles_visitante'], 'Gana',
        np.where(df['goles_local'] < df['goles_visitante'], 'Pierde', 'Empata')
    )
    return df

# Limpiar cada torneo y concatenar en una sola tabla
limpio18 = limpiar_mundial(d18, 2018)
limpio22 = limpiar_mundial(d22, 2022)
limpio26 = limpiar_mundial(d26, 2026)
partidos = pd.concat([limpio18, limpio22, limpio26], ignore_index=True)

## Validaciones obligatorias

La limpieza no termina cuando el código deja de producir errores. Debes comprobar invariantes.

In [ ]:
# Verificar invariantes: cantidad, duplicados, nulos, negativos y consistencia
print(f'2018: {len(limpio18)} partidos')
print(f'2022: {len(limpio22)} partidos')
print(f'2026: {len(limpio26)} partidos')
print(f'Total: {len(partidos)}')

assert len(limpio18) == 48, '2018 debe tener 48 partidos'
assert len(limpio22) == 48, '2022 debe tener 48 partidos'
assert len(limpio26) == 72, '2026 debe tener 72 partidos'
assert partidos['partido_id'].is_unique, 'Hay partido_id duplicados'

assert (partidos['goles_local'] >= 0).all(), 'Hay goles_local negativos'
assert (partidos['goles_visitante'] >= 0).all(), 'Hay goles_visitante negativos'

for col in ['equipo_local', 'equipo_visitante', 'goles_local', 'goles_visitante', 'grupo']:
    assert partidos[col].notna().all(), f'Hay nulos en {col}'

# Verificar que el marcador textual coincide con los goles numericos
def marcador_consistente(row):
    gl, gv = separar_marcador(row['marcador'])
    if pd.notna(gl):
        return gl == row['goles_local'] and gv == row['goles_visitante']
    return True

inconsistentes = ~partidos.apply(marcador_consistente, axis=1)
print(f'Marcadores inconsistentes: {inconsistentes.sum()}')


## Comparación de los Mundiales

In [ ]:
# Comparar usando tasas (goles por partido, % empates) no totales
# 2026 tiene 72 partidos vs 48 de 2018/2022: los totales enganan
comparacion = partidos.groupby('mundial').agg(
    partidos=('partido_id', 'count'),
    goles_totales=('goles_totales', 'sum'),
    empates=('resultado_local', lambda x: (x == 'Empata').sum()),
)
comparacion['goles_por_partido'] = comparacion['goles_totales'] / comparacion['partidos']
comparacion['porcentaje_empates'] = comparacion['empates'] / comparacion['partidos'] * 100
comparacion

# Grafico 1: goles totales vs goles por partido (misma escala no)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
comparacion['goles_totales'].plot(kind='bar', ax=axes[0])
axes[0].set_title('Goles totales')
axes[0].set_ylabel('Goles')
comparacion['goles_por_partido'].plot(kind='bar', ax=axes[1])
axes[1].set_title('Goles por partido')
axes[1].set_ylabel('Goles')
plt.tight_layout()
plt.show()

# Grafico 2: distribucion de resultados por torneo
fig, ax = plt.subplots(figsize=(8, 4))
pd.crosstab(partidos['mundial'], partidos['resultado_local']).plot(kind='bar', ax=ax)
ax.set_title('Distribución de resultados por mundial')
ax.set_ylabel('Partidos')
plt.tight_layout()
plt.show()


## Tabla por equipo

In [ ]:
# Convertir a formato largo: cada partido genera dos filas de equipo
local = partidos[['mundial', 'equipo_local', 'goles_local', 'goles_visitante', 'resultado_local']].copy()
local.columns = ['mundial', 'equipo', 'gf', 'gc', 'resultado']

# Para el visitante, el resultado se invierte (local Gana -> visitante Pierde)
visita = partidos[['mundial', 'equipo_visitante', 'goles_visitante', 'goles_local', 'resultado_local']].copy()
visita.columns = ['mundial', 'equipo', 'gf', 'gc', 'resultado']
visita['resultado'] = visita['resultado'].map({'Gana': 'Pierde', 'Pierde': 'Gana', 'Empata': 'Empata'})

equipos = pd.concat([local, visita], ignore_index=True)
equipos['puntos'] = equipos['resultado'].map({'Gana': 3, 'Empata': 1, 'Pierde': 0})

# Tabla de posiciones por torneo y equipo
tabla = equipos.groupby(['mundial', 'equipo']).agg(
    PJ=('resultado', 'count'),
    PG=('resultado', lambda x: (x == 'Gana').sum()),
    PE=('resultado', lambda x: (x == 'Empata').sum()),
    PP=('resultado', lambda x: (x == 'Pierde').sum()),
    GF=('gf', 'sum'),
    GC=('gc', 'sum'),
    PTS=('puntos', 'sum'),
).reset_index()
tabla['DG'] = tabla['GF'] - tabla['GC']
tabla['puntos_por_partido'] = tabla['PTS'] / tabla['PJ']
tabla


## Variables previas al partido

Para predecir no podemos utilizar datos ocurridos después del inicio. Crearemos promedios acumulados antes de cada encuentro.

In [ ]:
# Construir promedios acumulados ANTES de cada partido (evitar fuga)
def construir_variables_previas(partidos):
    rows = []
    stats = {}
    for mundial in sorted(partidos['mundial'].unique()):
        df_torneo = partidos[partidos['mundial'] == mundial].sort_values('fecha')
        for _, row in df_torneo.iterrows():
            local = row['equipo_local']
            visita = row['equipo_visitante']
            for equipo in (local, visita):
                if equipo not in stats:
                    stats[equipo] = {'pj': 0, 'pts': 0, 'gf': 0, 'gc': 0}
            row_data = {'partido_id': row['partido_id'], 'mundial': mundial}
            # Guardar promedios ANTES de actualizar (orden clave!)
            for prefijo, equipo in [('local', local), ('visita', visita)]:
                s = stats[equipo]
                row_data[f'{prefijo}_pts_prom_pre'] = s['pts'] / s['pj'] if s['pj'] > 0 else 0
                row_data[f'{prefijo}_gd_prom_pre'] = (s['gf'] - s['gc']) / s['pj'] if s['pj'] > 0 else 0
                row_data[f'{prefijo}_gf_prom_pre'] = s['gf'] / s['pj'] if s['pj'] > 0 else 0
            # Actualizar estadisticas con el resultado del partido actual
            gl = int(row['goles_local'])
            gv = int(row['goles_visitante'])
            stats[local]['pj'] += 1
            stats[local]['gf'] += gl
            stats[local]['gc'] += gv
            if gl > gv:
                stats[local]['pts'] += 3
            elif gl == gv:
                stats[local]['pts'] += 1
            stats[visita]['pj'] += 1
            stats[visita]['gf'] += gv
            stats[visita]['gc'] += gl
            if gv > gl:
                stats[visita]['pts'] += 3
            elif gv == gl:
                stats[visita]['pts'] += 1
            rows.append(row_data)
    return pd.DataFrame(rows)

features_df = construir_variables_previas(partidos)


## Entrenamiento y prueba

In [ ]:
# Solo variables conocidas antes del partido: jornada, promedios previos, anfitrion
features = [
    'jornada',
    'local_pts_prom_pre', 'visita_pts_prom_pre',
    'local_gd_prom_pre', 'visita_gd_prom_pre',
    'local_gf_prom_pre', 'visita_gf_prom_pre',
    'local_es_anfitrion', 'visitante_es_anfitrion'
]

# Entrenar con 2018+2022, probar con 2026
train_df = partidos[partidos['mundial'].isin(['2018', '2022'])].merge(features_df, on='partido_id')
test_df = partidos[partidos['mundial'] == '2026'].merge(features_df, on='partido_id')

X_train = train_df[features].fillna(0)  # primera jornada tiene promedios = 0
y_train = train_df['resultado_local']
X_test = test_df[features].fillna(0)
y_test = test_df['resultado_local']

# Linea base: predecir siempre la clase mas frecuente del entrenamiento
baseline = y_train.value_counts().idxmax()
baseline_acc = (y_test == baseline).mean()
print(f'Línea base (siempre "{baseline}"): {baseline_acc:.2%}')

# Arbol de decision con poda para evitar sobreajuste
clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f'Árbol de decisión: {accuracy_score(y_test, y_pred):.2%}')

# Matriz de confusion
ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test)
plt.show()


## Experimento de fuga de información

Agrega temporalmente `goles_local`, `goles_visitante` y `diferencia_goles` como variables. Si la precisión sube de forma extrema, explica por qué el modelo no está prediciendo realmente.

In [ ]:
# Experimento: agregar goles del partido como predictores (fuga de informacion)
leak_features = features + ['goles_local', 'goles_visitante', 'diferencia_goles']
X_train_leak = train_df[leak_features]
X_test_leak = test_df[leak_features]

clf_leak = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
clf_leak.fit(X_train_leak, y_train)
y_pred_leak = clf_leak.predict(X_test_leak)
print(f'Árbol con fuga (goles como predictores): {accuracy_score(y_test, y_pred_leak):.2%}')


## Reflexión final

**¿Qué problema de calidad fue el más difícil?**  
Las fechas: 7+ formatos distintos (ISO, dd/mm/yyyy, mm/dd/yyyy, dd-mm-yy, "Jun 16, 2018", serial Excel `46187`, `N/D`). El serial de Excel requirió conocer el origen (1899-12-30) y el rango del torneo para desambiguar formatos ambiguos como `06/12/2026` (¿junio o diciembre?).

**¿Qué decisión de limpieza podría cambiar los resultados?**  
La prioridad entre marcador textual vs goles separados al reparar. Si se usara primero el marcador siempre (incluso cuando los goles ya son válidos), podrían introducirse errores cuando el marcador usa "sin dato" pero los goles están bien. La regla aplicada fue: usar marcador solo si los goles faltan o son negativos.

**¿Por qué 2026 debe compararse mediante tasas?**  
2026 tiene 72 partidos de fase de grupos (48 equipos), mientras que 2018 y 2022 tienen 48 partidos (32 equipos). Comparar goles totales favorece a 2026 por tener más partidos, no por jugar mejor. Las tasas (goles por partido, % de empates) normalizan por cantidad de partidos.

**¿El árbol supera la línea base?**  
Apenas. Línea base (siempre "Pierde"): 25.00 %. Árbol: 30.56 %. La mejora es marginal (~5.5 pp), lo que sugiere que las variables pre-partido disponibles (jornada, promedios acumulados, anfitrión) tienen poco poder predictivo con este volumen de datos.

**¿Qué variables reales agregarías para mejorar una predicción?**  
Ranking FIFA previo al torneo, valor de mercado de la plantilla, historial de enfrentamientos directos, rendimiento en torneos anteriores, edad promedio del equipo, cantidad de jugadores en ligas top europeas, y distancia recorrida por la afición (factor localía real).

**¿Por qué un resultado de 100 % puede ser una señal de alarma?**  
100 % de precisión indica fuga de información (data leakage). En este caso, al incluir `goles_local`, `goles_visitante` y `diferencia_goles` como predictores, el árbol "adivina" el resultado porque estos datos ocurren durante o después del partido, no antes. Un modelo perfecto en datos reales casi siempre es señal de que algo está mal en la construcción de variables.